<a href="https://colab.research.google.com/github/abelunbound/fg_interactive_budget/blob/main/COPY_full_model_training_budget_project_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

from sklearn.model_selection import train_test_split

from sentence_transformers import CrossEncoder, InputExample
from torch.utils.data import DataLoader
import math


In [ ]:
with open("csv_version_labelled_dataset_mda_projects_priorities.csv", "r", encoding="utf-8", errors="replace") as f:
    for i, line in enumerate(f):
        if i in range(17124, 17,130):
            print(i, repr(line))

In [ ]:

df = pd.read_csv("csv_version_labelled_dataset_mda_projects_priorities.csv", engine='python', on_bad_lines='warn')

In [ ]:
len(df)

In [ ]:
df.columns.to_list()

In [ ]:
train_indices = []
test_indices = []

for agency in df['mda_code'].unique():
    agency_data = df[df['mda_code'] == agency]

    if len(agency_data) < 4:
        # Too few projects - all to training
        train_indices.extend(agency_data.index.tolist())
    else:
        # Check if stratification is viable
        can_stratify = (
            len(agency_data) >= 8 and
            agency_data['alignment'].value_counts().min() >= 2
        )

        train_agency, test_agency = train_test_split(
            agency_data,
            test_size=0.25,
            random_state=42,
            stratify=agency_data['alignment'] if can_stratify else None
        )
        train_indices.extend(train_agency.index.tolist())
        test_indices.extend(test_agency.index.tolist())

# train_df = df.loc[train_indices]
train_df = df.loc[train_indices].reset_index(drop=True)
temp_test_df = df.loc[test_indices]
# temp_test_df = df.loc[test_indices].reset_index(drop=True)


print(f"Train: {len(train_df)} projects, {train_df['mda_code'].nunique()} agencies")
print(f"Test: {len(temp_test_df)} projects, {temp_test_df['mda_code'].nunique()} agencies")
print(f"\nTrain alignment:\n{train_df['alignment'].value_counts()}")
print(f"\nTest alignment:\n{temp_test_df['alignment'].value_counts()}")

In [ ]:
test_indicies = []
val_indicies = []

for agency in temp_test_df['mda_code'].unique():
    agency_data = temp_test_df[temp_test_df['mda_code'] == agency]

    if len(agency_data) < 4:
        # Too few projects - all to training
        test_indicies.extend(agency_data.index.tolist())
    else:
        # Check if stratification is viable
        can_stratify = (
            len(agency_data) >= 8 and
            agency_data['alignment'].value_counts().min() >= 2
        )

        train_agency, test_agency = train_test_split(
            agency_data,
            test_size=0.40,
            random_state=42,
            stratify=agency_data['alignment'] if can_stratify else None
        )
        test_indicies.extend(train_agency.index.tolist())
        val_indicies.extend(test_agency.index.tolist())

val_df = df.loc[test_indicies]
val_df = df.loc [test_indicies].reset_index(drop=True)
test_df = df.loc[val_indicies]
test_df = df.loc[val_indicies].reset_index(drop=True)


print(f"Validation Set: {len(val_df)} projects, {val_df['mda_code'].nunique()} agencies")
print(f"Test Set: {len(test_df)} projects, {test_df['mda_code'].nunique()} agencies")
# print(f"\nTrain alignment:\n{train_df['alignment'].value_counts()}")
print(f"\nValidation alignment:\n{val_df['alignment'].value_counts()}")
print(f"\nTest alignment:\n{test_df['alignment'].value_counts()}")

In [ ]:
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")


In [ ]:
train_df.shape

In [ ]:


# ============================================
# 1. PREPARE DATA
# ============================================

# Convert to InputExample format
train_examples = []
for idx, row in train_df.iterrows():
    train_examples.append(
        InputExample(
            texts=[row['mandate'], row['project']],
            label=int(row['label'])
        )
    )

val_examples = []
for idx, row in val_df.iterrows():
    val_examples.append(
        InputExample(
            texts=[row['mandate'], row['project']],
            label=int(row['label'])
        )
    )


In [ ]:
# ============================================
# PREPARE TEST DATA
# ============================================

test_pairs = [
    (row['mandate'], row['project'])
    for idx, row in test_df.iterrows()
]
test_labels = test_df['label'].values


In [ ]:
len(test_labels)

In [ ]:
from sentence_transformers import CrossEncoder



# # ============================================
# # 2. LOAD BASE MODEL
# # ============================================

model = CrossEncoder(
    'cross-encoder/nli-deberta-v3-large',
    num_labels=2,
    max_length=1024,
    automodel_args={'ignore_mismatched_sizes': True}
)




In [ ]:
# ============================================
# 3. CONFIGURE TRAINING
# ============================================

train_dataloader = DataLoader(
    train_examples,
    shuffle=True,
    batch_size=4  # Adjust based on GPU memory
)

In [ ]:
print(train_dataloader)

In [ ]:
# ============================================
# 4. TRAIN -
# ============================================

from sentence_transformers.cross_encoder.evaluation import CEBinaryClassificationEvaluator

# This block is needed for evaluating the best model to save
evaluator = CEBinaryClassificationEvaluator(
    sentence_pairs=test_pairs,
    labels=test_labels.tolist(),
    name='budget-eval'
)

num_epochs = 3
warmup_steps = math.ceil(len(train_dataloader) * num_epochs * 0.1)  # 10% warmup


model.fit(
    train_dataloader=train_dataloader,
    epochs=num_epochs,
    warmup_steps=warmup_steps,
    evaluator=evaluator, #without evaluator no models folder created
    evaluation_steps=500o,
    output_path='./models/deberta-v3-budget-classifier',
    save_best_model=True,
    show_progress_bar=True
)

In [ ]:
# ============================================
# 4. TRAIN - temporarily commented out (code doesn't save to .models/ folder)
# ============================================

# num_epochs = 1
# warmup_steps = math.ceil(len(train_dataloader) * num_epochs * 0.1)  # 10% warmup

# model.fit(
#     train_dataloader=train_dataloader,
#     epochs=num_epochs,
#     warmup_steps=warmup_steps,
#     output_path='./models/deberta-v3-budget-classifier',
#     save_best_model=True,
#     show_progress_bar=True
# )

# print("Training complete!")

In [ ]:
# Option 1: Basic Evaluation

from sentence_transformers import CrossEncoder
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# ============================================
# LOAD FINE-TUNED MODEL
# ============================================

model = CrossEncoder('./models/deberta-v3-budget-classifier')

# ============================================
# PREPARE TEST DATA
# ============================================

test_pairs = [
    (row['mandate'], row['project'])
    for idx, row in test_df.iterrows()
]
test_labels = test_df['label'].values

# ============================================
# PREDICT
# ============================================

# Get logits
predictions_logits = model.predict(test_pairs, show_progress_bar=True)

# Convert to probabilities
def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp_x / exp_x.sum(axis=1, keepdims=True)

predictions_probs = softmax(predictions_logits)

# Get predicted classes
predictions = np.argmax(predictions_logits, axis=1)

# ============================================
# EVALUATION METRICS
# ============================================

print(classification_report(
    test_labels,
    predictions,
    target_names=['Outside Mandate (0)', 'Within Mandate (1)']
))

# Output:
#                        precision    recall  f1-score   support
# Within Mandate (0)         0.92      0.95      0.93      4200
# Outside Mandate (1)        0.78      0.68      0.73       800
#           accuracy                             0.90      5000

In [ ]:
# Option 2: Comprehensive Evaluation


from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    roc_auc_score,
    roc_curve
)
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================
# COMPUTE METRICS
# ============================================

def evaluate_model(y_true, y_pred, y_probs):
    """Comprehensive evaluation"""

    # Basic metrics
    acc = accuracy_score(y_true, y_pred)

    # Per-class metrics
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, average=None
    )

    # Weighted averages
    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        y_true, y_pred, average='weighted'
    )

    # Class 1 specific (outside mandate - what you care about)
    precision_class1 = precision[1]
    recall_class1 = recall[1]
    f1_class1 = f1[1]

    # False positive rate
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    fpr = fp / (fp + tn)

    # AUC
    auc = roc_auc_score(y_true, y_probs[:, 1])

    # Print results
    print("=" * 60)
    print("OVERALL METRICS")
    print("=" * 60)
    print(f"Accuracy:           {acc:.4f}")
    print(f"F1 (weighted):      {f1_weighted:.4f}")
    print(f"Precision (weighted): {precision_weighted:.4f}")
    print(f"Recall (weighted):    {recall_weighted:.4f}")
    print(f"AUC-ROC:            {auc:.4f}")
    print()

    print("=" * 60)
    print("CLASS 0: Outside Mandate (Critical for Your Task)")
    print("=" * 60)
    print(f"Precision:          {precision[0]:.4f} ← How often flags are correct")
    print(f"Recall:             {recall[0]:.4f} ← % of violations caught")
    print(f"F1-Score:           {f1[0]:.4f}")
    print(f"Support:            {support[0]}")
    print()

    print("=" * 60)
    print("CLASS 1: Within Mandate")
    print("=" * 60)
    print(f"Precision:          {precision_class1:.4f}  ← How often flags are correct")
    print(f"Recall:             {recall_class1:.4f}  ")
    print(f"F1-Score:           {f1_class1:.4f}")
    print(f"Support:            {support[1]}")
    print()

    print("=" * 60)
    print("ERROR ANALYSIS")
    print("=" * 60)
    print(f"False Positive Rate: {fpr:.4f}  ← % good projects wrongly flagged")
    print(f"False Positives:     {fp} / {fp + tn} Class 0 examples")
    print(f"False Negatives:     {fn} / {fn + tp} Class 1 examples")
    print()

    return {
        'accuracy': acc,
        'f1': f1_weighted,
        'precision_class1': precision_class1,
        'recall_class1': recall_class1,
        'f1_class1': f1_class1,
        'fpr': fpr,
        'auc': auc,
        'confusion_matrix': cm
    }

# Run evaluation
metrics = evaluate_model(test_labels, predictions, predictions_probs)

In [ ]:
# Option 3: Visualisation

import matplotlib.pyplot as plt
import seaborn as sns

# ============================================
# 1. CONFUSION MATRIX
# ============================================

cm = confusion_matrix(test_labels, predictions)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Outside (0)', 'Within (1)'],
    yticklabels=['Outside (0)', 'Within (1)']
)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=300)
plt.show()

# ============================================
# 2. ROC CURVE
# ============================================

fpr, tpr, thresholds = roc_curve(test_labels, predictions_probs[:, 1])
auc = roc_auc_score(test_labels, predictions_probs[:, 1])

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'AUC = {auc:.3f}')
plt.plot([0, 1], [0, 1], 'k--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('roc_curve.png', dpi=300)
plt.show()

# ============================================
# 3. CONFIDENCE DISTRIBUTION
# ============================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Class 0 confidence
class0_mask = test_labels == 0
class0_confidence = predictions_probs[class0_mask, 0]

axes[0].hist(class0_confidence, bins=50, color='green', alpha=0.7)
axes[0].set_xlabel('Confidence')
axes[0].set_ylabel('Count')
axes[0].set_title('Class 0 (Outside Mandate) - Confidence Distribution')
axes[0].axvline(0.7, color='red', linestyle='--', label='Threshold')
axes[0].legend()

# Class 1 confidence
class1_mask = test_labels == 1
class1_confidence = predictions_probs[class1_mask, 1]

axes[1].hist(class1_confidence, bins=50, color='orange', alpha=0.7)
axes[1].set_xlabel('Confidence')
axes[1].set_ylabel('Count')
axes[1].set_title('Class 1 (Within Mandate) - Confidence Distribution')
axes[1].axvline(0.7, color='red', linestyle='--', label='Threshold')
axes[1].legend()

plt.tight_layout()
plt.savefig('confidence_distribution.png', dpi=300)
plt.show()

# ============================================
# 4. PRECISION-RECALL CURVE
# ============================================

from sklearn.metrics import precision_recall_curve, average_precision_score

precision_curve, recall_curve, thresholds_pr = precision_recall_curve(
    test_labels, predictions_probs[:, 1]
)
avg_precision = average_precision_score(test_labels, predictions_probs[:, 1])

plt.figure(figsize=(8, 6))
plt.plot(recall_curve, precision_curve, label=f'AP = {avg_precision:.3f}')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve (Class 1: Within Mandate)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('precision_recall_curve.png', dpi=300)
plt.show()

In [ ]:
# ============================================
# ANALYZE MISCLASSIFICATIONS
# ============================================

test_df_copy = test_df.copy()
test_df_copy['predicted'] = predictions
test_df_copy['confidence_class0'] = predictions_probs[:, 0]
test_df_copy['confidence_class1'] = predictions_probs[:, 1]
test_df_copy['correct'] = test_df_copy['label'] == test_df_copy['predicted']

# False positives (flagged as outside, but actually within)
false_positives = test_df_copy[
    (test_df_copy['label'] == 0) & (test_df_copy['predicted'] == 1)
].sort_values('confidence_class1', ascending=False)

print("=" * 60)
print("TOP 10 FALSE POSITIVES (Wrongly Flagged)")
print("=" * 60)
for idx, row in false_positives.head(10).iterrows():
    print(f"\nMandate: {row['mandate'][:100]}...")
    print(f"Project: {row['project'][:100]}...")
    print(f"Confidence: {row['confidence_class1']:.2%}")
    print("-" * 60)

# False negatives (should be flagged, but missed)
false_negatives = test_df_copy[
    (test_df_copy['label'] == 1) & (test_df_copy['predicted'] == 0)
].sort_values('confidence_class0', ascending=False)

print("\n" + "=" * 60)
print("TOP 10 FALSE NEGATIVES (Missed Violations)")
print("=" * 60)
for idx, row in false_negatives.head(10).iterrows():
    print(f"\nMandate: {row['mandate'][:100]}...")
    print(f"Project: {row['project'][:100]}...")
    print(f"Confidence: {row['confidence_class0']:.2%}")
    print("-" * 60)

# Save full error analysis
test_df_copy.to_csv('test_results_with_predictions.csv', index=False)

In [ ]:
# ============================================
# EVALUATE BY AGENCY
# ============================================

# Assuming you have MDA codes in your test data
test_df_copy['mda_code'] = test_df['mda_code']  # Add if available

agency_performance = test_df_copy.groupby('mda_code').apply(
    lambda group: pd.Series({
        'total': len(group),
        'accuracy': (group['label'] == group['predicted']).mean(),
        'class1_count': (group['label'] == 1).sum(),
        'class1_recall': (
            (group['label'] == 1) & (group['predicted'] == 1)
        ).sum() / max((group['label'] == 1).sum(), 1)
    })
).sort_values('accuracy')

print("Worst Performing Agencies:")
print(agency_performance.head(10))

print("\nBest Performing Agencies:")
print(agency_performance.tail(10))

# Evaluate by Agency

agency_performance.to_csv('agency_performance.csv', index=False)

In [ ]:
# Check for NaN in either column
print(val_df['mandate'].isna().sum(), "missing mandates")
print(val_df['project'].isna().sum(), "missing projects")

# See which rows
print(val_df[val_df['mandate'].isna() | val_df['project'].isna()])

In [ ]:
# Option 1 — drop rows with missing values
val_df = val_df.dropna(subset=['mandate', 'project'])

In [ ]:
# ============================================
# MODEL VALIDATION
# ============================================

from sentence_transformers import CrossEncoder
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# ============================================
# LOAD FINE-TUNED MODEL
# ============================================

model = CrossEncoder('./models/deberta-v3-budget-classifier')

# ============================================
# PREPARE VALIDATION DATA
# ============================================

validation_pairs = [
    (row['mandate'], row['project'])
    for idx, row in val_df.iterrows()
]
validation_labels = val_df['label'].values

# ============================================
# PREDICT
# ============================================

# Get logits
predictions_logits = model.predict(validation_pairs, show_progress_bar=True)

# Convert to probabilities
def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp_x / exp_x.sum(axis=1, keepdims=True)

predictions_probs = softmax(predictions_logits)

# Get predicted classes
predictions = np.argmax(predictions_logits, axis=1)

# ============================================
# EVALUATION METRICS
# ============================================

print(classification_report(
    validation_labels,
    predictions,
    target_names=['Outside Mandate (0)', 'Within Mandate (1)']
))



In [ ]:
# ============================================
# INFERENCE ON 2026 BUDGET
# ============================================


from sentence_transformers import CrossEncoder
import numpy as np
import pandas as pd

# ============================================
# LOAD FINE-TUNED MODEL
# ============================================

model = CrossEncoder('./models/deberta-v3-budget-classifier')


# Load your 2026 budget (approx. 20,000 projects?)
budget_2026 = pd.read_csv('budget_2026_test.csv')

# Prepare pairs
pairs = [
    (row['mandate'], row['project'])
    for idx, row in budget_2026.iterrows()
]

# Batch predict
batch_size = 32
all_predictions = []

for i in range(0, len(pairs), batch_size):
    batch = pairs[i:i+batch_size]
    preds = model.predict(batch)
    all_predictions.extend(preds)

all_predictions = np.array(all_predictions)

# Softmax for probabilities
probs = np.exp(all_predictions) / np.exp(all_predictions).sum(axis=1, keepdims=True)

# Add results to dataframe
budget_2026['predicted_class'] = np.argmax(all_predictions, axis=1)
budget_2026['prob_outside'] = probs[:, 0]
budget_2026['prob_within'] = probs[:, 1]

# Flag high-confidence violations
budget_2026['flag_for_review'] = (
    (budget_2026['predicted_class'] == 1) &
    (budget_2026['prob_outside'] > 0.7)
)

print(f"Total projects flagged: {budget_2026['flag_for_review'].sum()}")
print(f"Percentage flagged: {budget_2026['flag_for_review'].mean():.2%}")

# Export results
budget_2026.to_csv('budget_2026_classified.csv', index=False)

In [ ]:
import shutil
from google.colab import drive, files
import os


# ============================================
# X. ZIP EVERYTHING AND DOWNLOAD
# ============================================
shutil.make_archive('/content/fg_budget_modelv2', 'zip', '/content/models', '.')
# files.download('/content/session_export.zip')
# print("Download started!")



In [ ]:
from huggingface_hub import login, upload_folder

# (optional) Login with your Hugging Face credentials
login()

# Push your model files
upload_folder(
    folder_path="./models/deberta-v3-budget-classifier",
    repo_id="abelakeni/pfmtools-deberta-v3-fg-budget-v3-classifier",
    repo_type="model"
)


In [ ]:
from google.colab import drive
import zipfile, os

# # Mount your Drive
# drive.mount('/content/drive')


In [ ]:

# Replace with the actual path to your file in Drive
zip_file_path = '/content/drive/MyDrive/fg_budget_modelv2.zip'
extraction_path = '/content/models'

os.makedirs(extraction_path, exist_ok=True)

with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extraction_path)

print("Done!")